# Session 2: Prompt Engineering for Agentic Behaviors (Student Notebook)

## Learning Objectives

By the end of this session, you will be able to:

1. **Craft effective system prompts** that define agent personas, capabilities, and constraints
2. **Implement Chain-of-Thought (CoT) and ReAct prompting** patterns to improve reasoning quality
3. **Generate structured outputs** (JSON mode) for reliable downstream processing
4. **Manage multi-turn conversations** with context accumulation and summarization strategies

In [ ]:
# Imports and Setup
import openai
import json
import os

# Ensure your OPENAI_API_KEY is set in your environment variables
# os.environ["OPENAI_API_KEY"] = "your-api-key-here"

client = openai.OpenAI()
print("Setup complete. OpenAI client initialized.")

---
## Demos

Follow along with these demos. Run each cell and observe the outputs carefully.

### Demo 1: Zero-Shot vs. Few-Shot Prompting

**Zero-shot** prompting asks the model to perform a task with no examples.  
**Few-shot** prompting provides a handful of examples so the model can learn the expected format and behavior.

We will compare both approaches on a sentiment classification task.

In [ ]:
# Demo 1: Zero-Shot vs. Few-Shot Prompting

# Zero-shot: no examples provided
zero_shot_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Classify the sentiment of this review as positive, negative, or neutral: 'The product works fine but the delivery was terrible.'"}]
)
print("Zero-shot:", zero_shot_response.choices[0].message.content)

print("\n" + "="*60 + "\n")

# Few-shot: provide labeled examples first
few_shot_messages = [
    {"role": "system", "content": "You are a sentiment classifier. Respond with only: POSITIVE, NEGATIVE, or NEUTRAL."},
    {"role": "user", "content": "Review: 'Absolutely loved this product!'"},
    {"role": "assistant", "content": "POSITIVE"},
    {"role": "user", "content": "Review: 'Worst purchase ever, total waste of money.'"},
    {"role": "assistant", "content": "NEGATIVE"},
    {"role": "user", "content": "Review: 'The product works fine but the delivery was terrible.'"}
]
few_shot_response = client.chat.completions.create(model="gpt-4o-mini", messages=few_shot_messages)
print("Few-shot:", few_shot_response.choices[0].message.content)

**Observation:** Notice how the zero-shot response may include extra explanation, while the few-shot response follows the exact format demonstrated in the examples.

### Demo 2: Chain-of-Thought Prompting

Chain-of-Thought (CoT) prompting encourages the model to show its reasoning process step by step, which improves accuracy on math, logic, and multi-step problems.

In [ ]:
# Demo 2: Chain-of-Thought Prompting

problem = "A store has 45 apples. They sell 60% in the morning and half of the remainder in the afternoon. How many are left?"

# Without CoT
response_no_cot = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": problem}]
)
print("Without CoT:")
print(response_no_cot.choices[0].message.content)

print("\n" + "="*60 + "\n")

# With explicit CoT
cot_prompt = f"{problem}\n\nLet's solve this step by step:\n1. First, calculate how many were sold in the morning\n2. Then, find the remainder\n3. Calculate how many were sold in the afternoon\n4. Find the final count"
response_cot = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": cot_prompt}]
)
print("With CoT:")
print(response_cot.choices[0].message.content)

**Observation:** The CoT version explicitly walks through each calculation, making it easier to verify correctness and debug any errors in reasoning.

### Demo 3: Role-Based Prompting for Agentic Personas

By changing the system prompt, we can create entirely different agent personas that approach the same question from different perspectives. This is foundational for building specialized agents.

In [ ]:
# Demo 3: Role-Based Prompting for Agentic Personas

personas = {
    "Data Analyst": "You are a senior data analyst. You approach every question with data-driven thinking, ask for metrics, and suggest analytical frameworks.",
    "Security Expert": "You are a cybersecurity expert. You evaluate everything through the lens of security risks, vulnerabilities, and compliance requirements.",
    "Product Manager": "You are a product manager. You think about user needs, business value, prioritization, and stakeholder communication."
}
question = "We want to add a new feature to our web application that allows users to upload files."

for role, system_prompt in personas.items():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ],
        max_tokens=200
    )
    print(f"\n{'='*50}\n{role}:\n{response.choices[0].message.content}")

**Observation:** Each persona focuses on different aspects of the same question -- data metrics, security risks, or user/business impact. This demonstrates how system prompts shape agent behavior.

### Demo 4: Structured Output Generation (JSON Mode)

When building agentic systems, we often need the LLM output to be machine-readable. JSON mode ensures the response is valid JSON, making downstream processing reliable.

In [ ]:
# Demo 4: Structured Output Generation (JSON Mode)

text = "John Smith is a 35-year-old software engineer from San Francisco. He has 10 years of experience and specializes in Python and machine learning."

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Extract the person's information and return it as a JSON object with keys: name, age, occupation, location, experience_years, skills (as a list)."},
        {"role": "user", "content": text}
    ],
    response_format={"type": "json_object"}
)

parsed = json.loads(response.choices[0].message.content)
print("Parsed JSON output:")
print(json.dumps(parsed, indent=2))

# Demonstrate that we can access individual fields programmatically
print(f"\nName: {parsed.get('name')}")
print(f"Skills: {', '.join(parsed.get('skills', []))}")

**Observation:** With `response_format={"type": "json_object"}`, the output is guaranteed to be valid JSON, which we can reliably parse and use in code.

### Demo 5: Multi-Turn Conversation Management

Agents need to maintain context across multiple exchanges. This demo shows how to build a conversation manager that accumulates context and tracks history.

In [ ]:
# Demo 5: Multi-Turn Conversation Management

class ConversationManager:
    def __init__(self, system_prompt, model="gpt-4o-mini"):
        self.model = model
        self.messages = [{"role": "system", "content": system_prompt}]
        self.client = openai.OpenAI()
    
    def send(self, user_message):
        self.messages.append({"role": "user", "content": user_message})
        response = self.client.chat.completions.create(
            model=self.model,
            messages=self.messages,
            max_tokens=300
        )
        assistant_msg = response.choices[0].message.content
        self.messages.append({"role": "assistant", "content": assistant_msg})
        return assistant_msg
    
    def get_history_length(self):
        return len(self.messages)

# Demo the conversation
conv = ConversationManager("You are a helpful travel planning assistant.")

print("User: I want to plan a trip to Japan.")
print("Assistant:", conv.send("I want to plan a trip to Japan."))

print("\n" + "-"*50)
print("\nUser: What's the best time to visit?")
print("Assistant:", conv.send("What's the best time to visit?"))

print("\n" + "-"*50)
print("\nUser: Can you summarize what we've discussed so far?")
print("Assistant:", conv.send("Can you summarize what we've discussed so far?"))

print(f"\nConversation history: {conv.get_history_length()} messages")

**Observation:** The assistant remembers context from earlier turns -- it knows we are discussing Japan without being reminded. The conversation history grows with each exchange.

---
## Tasks

Now it is your turn! Complete the following tasks by filling in the `TODO` sections. Use the demo code above as reference.

### Task 1: Design Effective System Prompts for a Research Agent

Create a system prompt for a "Research Agent" and test it with a research question about renewable energy.

In [ ]:
# Task 1: Design Effective System Prompts for a Research Agent

# TODO: Create a system prompt for a "Research Agent" that:
# 1. Defines the agent's role and expertise
# 2. Specifies how it should approach research questions
# 3. Defines output format (structured with sections: Summary, Key Findings, Sources, Confidence Level)
# 4. Includes constraints (acknowledge uncertainty, cite reasoning)
#
# Then test it with a research question about renewable energy.
#
# Hint: Good system prompts are 100-300 words
# Hint: Include specific output formatting instructions
# Hint: Add behavioral guidelines (e.g., "If uncertain, say so explicitly")

research_agent_prompt = ""  # YOUR CODE HERE

def ask_research_agent(question):
    # YOUR CODE HERE
    pass

# Test it
# ask_research_agent("What are the latest trends in renewable energy storage?")

### Task 2: Implement ReAct-Style Prompting (Reason + Act)

Implement a ReAct-style prompt that interleaves Thought, Action, and Observation steps to solve a question.

In [ ]:
# Task 2: Implement ReAct-Style Prompting (Reason + Act)

# TODO: Implement a ReAct-style prompt that follows the pattern:
# Thought: [reasoning about what to do]
# Action: [what action to take]
# Observation: [result of the action]
# ... (repeat)
# Final Answer: [conclusion]
#
# Create a prompt that solves: "Find the population of the capital of France"
# The agent should reason through: identify capital -> look up population -> answer
#
# Hint: Define available "actions" in the system prompt (e.g., Search, Calculate, Lookup)
# Hint: Structure the system prompt with the ReAct format template
# Hint: The LLM simulates the actions since we don't have real tools yet

def react_agent(question):
    # YOUR CODE HERE
    pass

# Test it
# react_agent("What is the population of the capital of France?")

### Task 3: Create a Reusable Prompt Template Library

Build a `PromptTemplate` class with variable placeholders, validation, and a library of pre-built templates.

In [ ]:
# Task 3: Create a Reusable Prompt Template Library

# TODO: Build a PromptTemplate class that:
# 1. Stores template strings with {variable} placeholders
# 2. Has a .format() method to fill in variables
# 3. Has a .validate() method to check all variables are provided
# 4. Includes at least 3 pre-built templates: summarization, classification, extraction
#
# Hint: Use Python string .format() or f-strings
# Hint: Use re.findall(r'\{(\w+)\}', template) to find placeholders
# Hint: Raise an error if required variables are missing

import re

class PromptTemplate:
    def __init__(self, name, template, description=""):
        # YOUR CODE HERE
        pass
    
    def format(self, **kwargs):
        # YOUR CODE HERE
        pass
    
    def validate(self, **kwargs):
        # YOUR CODE HERE
        pass

# Create template library
# summarize_template = PromptTemplate(...)
# classify_template = PromptTemplate(...)
# extract_template = PromptTemplate(...)

### Task 4: Build a Multi-Step Reasoning Agent with Tool Descriptions

Build an agent that uses structured JSON output to decide which tools to call, simulates tool execution, and produces a final answer.

In [ ]:
# Task 4: Build a Multi-Step Reasoning Agent with Tool Descriptions

# TODO: Build an agent that:
# 1. Has a system prompt describing available tools (calculator, search, weather)
# 2. Receives a complex question requiring multiple steps
# 3. Uses structured output (JSON) to specify which tool to call at each step
# 4. Processes the "tool results" and generates a final answer
#
# Hint: Define tools as a list of dicts with name, description, parameters
# Hint: Ask the LLM to respond with JSON: {"tool": "name", "input": "..."}
# Hint: Simulate tool responses and feed them back

tools = [
    {"name": "calculator", "description": "Performs mathematical calculations", "parameters": "expression (string)"},
    {"name": "search", "description": "Searches for factual information", "parameters": "query (string)"},
    {"name": "weather", "description": "Gets current weather for a location", "parameters": "city (string)"}
]

class ToolAgent:
    def __init__(self):
        # YOUR CODE HERE
        pass
    
    def process(self, question):
        # YOUR CODE HERE
        pass

# Test: "What's 15% tip on a $85 dinner, and what's the weather like in New York?"

---
## Summary

### Key Takeaways

1. **Zero-shot vs. Few-shot**: Few-shot prompting provides examples that guide the model toward consistent format and behavior. Use it when you need predictable, structured responses.

2. **Chain-of-Thought (CoT)**: Explicitly asking for step-by-step reasoning improves accuracy on complex tasks. CoT is especially valuable for math, logic, and multi-step problems.

3. **Role-Based Prompting**: System prompts are the primary mechanism for defining agent personas. A well-crafted system prompt shapes the entire behavior profile of an agent.

4. **Structured Outputs**: JSON mode ensures machine-readable responses, which is critical for agentic workflows where outputs feed into downstream code or other agents.

5. **Multi-Turn Management**: Maintaining conversation history enables context-aware interactions, but requires careful management of the context window as conversations grow.

### Looking Ahead

These prompt engineering techniques form the foundation for building agentic systems. In the next session, we will explore how to evaluate and measure the quality of agent outputs systematically.